In [1]:
from pathlib import Path
from pypdf import PdfReader
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer


c:\Users\ZINEB\Desktop\rag-document-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
data_folder = Path("../data")

print("Dossier data :", data_folder.resolve())

Dossier data : C:\Users\ZINEB\Desktop\rag-document-assistant\data


In [47]:
pdf_files = list(data_folder.glob("*.pdf"))

print("Nombre de PDF trouvés :", len(pdf_files))

for pdf in pdf_files:
    print(pdf.name)

Nombre de PDF trouvés : 3
pca.pdf
retrieval.pdf
unspervisedlearning.pdf


In [48]:
documents = []

for pdf_path in pdf_files:
    print("Lecture de :", pdf_path.name)

    reader = PdfReader(pdf_path)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text:
            documents.append({
                "source": pdf_path.name,
                "page": page_number,
                "text": text
            })

Lecture de : pca.pdf
Lecture de : retrieval.pdf


Ignoring wrong pointing object 198 0 (offset 0)


Lecture de : unspervisedlearning.pdf


In [49]:
print("Nombre total de pages extraites :", len(documents))

Nombre total de pages extraites : 85


In [50]:
documents[0]

{'source': 'pca.pdf',
 'page': 1,
 'text': 'A Tutorial on Principal Component Analysis\nJonathon Shlens∗\nGoogle Research\nMountain View, CA 94043\n(Dated: April 7, 2014; Version 3.02)\nPrincipal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used\nbut (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This\nmanuscript focuses on building a solid intuition for how and why principal component analysis works. This\nmanuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics behind PCA. This\ntutorial does not shy away from explaining the ideas informally, nor does it shy away from the mathematics. The\nhope is that by addressing both aspects, readers of all levels will be able to gain a better understanding of PCA as\nwell as the when, the how and the why of applying this technique.\nI. INTRODUCTION\nPrincipal component analysis (PCA) is a standard tool in mod-\ner

In [51]:
print("Source :", documents[0]["source"])
print("Page :", documents[0]["page"])
print("\nTexte :")
print(documents[0]["text"][:500])

Source : pca.pdf
Page : 1

Texte :
A Tutorial on Principal Component Analysis
Jonathon Shlens∗
Google Research
Mountain View, CA 94043
(Dated: April 7, 2014; Version 3.02)
Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
manuscript focuses on building a solid intuition for how and why principal component analysis works. This
manuscript crystallizes this knowledge by


In [52]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [53]:
sample_text = documents[0]["text"]

test_chunks = split_text(sample_text)

print("Nombre de chunks :", len(test_chunks))

Nombre de chunks : 13


In [54]:
print(test_chunks[0])

A Tutorial on Principal Component Analysis
Jonathon Shlens∗
Google Research
Mountain View, CA 94043
(Dated: April 7, 2014; Version 3.02)
Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
manuscript focuses on building a solid intuition for how and why principal component analysis works. This
manuscript crystallizes this knowledge by


In [55]:
print(test_chunks[1])

n for how and why principal component analysis works. This
manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics behind PCA. This
tutorial does not shy away from explaining the ideas informally, nor does it shy away from the mathematics. The
hope is that by addressing both aspects, readers of all levels will be able to gain a better understanding of PCA as
well as the when, the how and the why of applying this technique.
I. INTRODUCTION
Principal component an


In [56]:
chunks = []

for document in documents:
    page_chunks = split_text(document["text"])

    for chunk in page_chunks:
        chunks.append({
            "source": document["source"],
            "page": document["page"],
            "text": chunk
        })

In [57]:
chunks = []

for document in documents:
    page_chunks = split_text(document["text"])

    for chunk in page_chunks:
        chunks.append({
            "source": document["source"],
            "page": document["page"],
            "text": chunk
        })

In [58]:
print("Nombre total de chunks :", len(chunks))

Nombre total de chunks : 398


In [59]:
chunks[0]

{'source': 'pca.pdf',
 'page': 1,
 'text': 'A Tutorial on Principal Component Analysis\nJonathon Shlens∗\nGoogle Research\nMountain View, CA 94043\n(Dated: April 7, 2014; Version 3.02)\nPrincipal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used\nbut (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This\nmanuscript focuses on building a solid intuition for how and why principal component analysis works. This\nmanuscript crystallizes this knowledge by'}

In [60]:

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ZINEB\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ZINEB\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [61]:


def split_text_by_sentences(text, max_chars=500):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [62]:
sample_text = documents[0]["text"]
nltk.download("punkt_tab")

sentence_chunks = split_text_by_sentences(sample_text)

print("Nombre de chunks :", len(sentence_chunks))

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ZINEB\AppData\Roaming\nltk_data...


Nombre de chunks : 12


[nltk_data]   Package punkt_tab is already up-to-date!


In [63]:
print(sentence_chunks[0])

A Tutorial on Principal Component Analysis
Jonathon Shlens∗
Google Research
Mountain View, CA 94043
(Dated: April 7, 2014; Version 3.02)
Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
manuscript focuses on building a solid intuition for how and why principal component analysis works.


In [64]:
print(sentence_chunks[1])

This
manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics behind PCA. This
tutorial does not shy away from explaining the ideas informally, nor does it shy away from the mathematics. The
hope is that by addressing both aspects, readers of all levels will be able to gain a better understanding of PCA as
well as the when, the how and the why of applying this technique. I.


**SentenceTransformers**

In [2]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\ZINEB\Desktop\rag-document-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ZINEB\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ZINEB\Desktop\rag-document-assistant\.venv\Lib\site-packages\huggingf

In [3]:
sentence = "Principal Component Analysis reduces dimensionality."

embedding = embedding_model.encode(sentence)

print(embedding.shape)
print(embedding[:10])

(384,)
[ 0.02052276 -0.01720105  0.06164395 -0.00636335  0.05441492  0.0376993
 -0.04214282 -0.0890014   0.02084276  0.02088071]


In [4]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Nombre de chunks :", len(chunk_texts))
print("Shape des embeddings :", embeddings.shape)

NameError: name 'chunks' is not defined